In [1]:
# ===================================================
# CAPSDAC Vendor / Site / Program 3–5 Month Recursive Forecast
# UPDATED:
# - Adds LEAName and PreschoolName into all forecast outputs
# - ChildFamily still drives the population count
# - ChildEnrollment still provides supplemental fields
# - Same output locations preserved
# ===================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from functools import reduce

ROOT = "abfss://yzhang@cdewarehouseuser.dfs.core.windows.net/capsdac2_silver_parquet"

CHILD_FAMILY_PATH = f"{ROOT}/ChildFamily_MonthlySnapshots"
CHILD_ENROLLMENT_PATH = f"{ROOT}/ChildEnrollment_MonthlySnapshots"

START_REPORT_MONTH = 202507
END_REPORT_MONTH = 202606

OUTPUT_ROOT = f"{ROOT}/ml_outputs/vendor_site_forecast"

FORECAST_DETAIL_OUT = f"{OUTPUT_ROOT}/forecast_5m_vendor_site_program"
FORECAST_TOTAL_OUT = f"{OUTPUT_ROOT}/forecast_summary_total"
FORECAST_VENDOR_OUT = f"{OUTPUT_ROOT}/forecast_summary_vendor"
FORECAST_SITE_OUT = f"{OUTPUT_ROOT}/forecast_summary_site"

def read_multi_month(path):
    df = spark.read.option("basePath", path).parquet(path)

    if "ReportMonthKey" in df.columns:
        df = df.withColumn("ReportMonthNum", F.col("ReportMonthKey").cast("int"))
    elif "ReportMonth" in df.columns:
        df = df.withColumn("ReportMonthNum", F.col("ReportMonth").cast("int"))
    elif "ReportMonthNum" in df.columns:
        df = df.withColumn("ReportMonthNum", F.col("ReportMonthNum").cast("int"))
    else:
        raise ValueError(f"{path} missing report month column")

    return df.filter(
        (F.col("ReportMonthNum") >= int(START_REPORT_MONTH)) &
        (F.col("ReportMonthNum") <= int(END_REPORT_MONTH))
    )

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def select_or_default(df, output_col, candidates, default_value="UNKNOWN"):
    source_col = first_existing_col(df, candidates)

    if source_col is None:
        return F.lit(default_value).alias(output_col)

    return F.col(source_col).cast("string").alias(output_col)

def add_months_yyyymm(col_name, months_to_add):
    return F.date_format(
        F.add_months(
            F.to_date(
                F.concat(F.col(col_name).cast("string"), F.lit("01")),
                "yyyyMMdd"
            ),
            months_to_add
        ),
        "yyyyMM"
    ).cast("int")

# ---------------------------------------------------
# Read data
# ---------------------------------------------------

cf = read_multi_month(CHILD_FAMILY_PATH)
ce = read_multi_month(CHILD_ENROLLMENT_PATH)

print("ChildFamily rows:", cf.count())
print("ChildEnrollment rows:", ce.count())

# ---------------------------------------------------
# ChildFamily driver with names
# ---------------------------------------------------

cf_base = (
    cf
    .select(
        F.col("CSPPID").cast("string").alias("CSPPID"),
        F.col("ReportMonthNum").cast("int").alias("ReportMonthNum"),

        select_or_default(
            cf,
            "VendorNumber",
            ["VendorNumber", "VendorId", "VendorID", "ReportingLEA"],
            "UNKNOWN"
        ),

        select_or_default(
            cf,
            "LEAName",
            [
                "LEAName",
                "VendorName",
                "VendorLongName",
                "AgencyName",
                "ReportingLEAName"
            ],
            "UNKNOWN"
        ),

        select_or_default(
            cf,
            "PreschoolCDSCode",
            [
                "PreschoolCDSCode",
                "PreschoolCode",
                "CDSCode"
            ],
            "UNKNOWN"
        ),

        select_or_default(
            cf,
            "PreschoolName",
            [
                "PreschoolName",
                "SchoolName",
                "SiteName",
                "PreschoolSiteName"
            ],
            "UNKNOWN"
        )
    )
    .dropDuplicates([
        "CSPPID",
        "ReportMonthNum",
        "VendorNumber",
        "LEAName",
        "PreschoolCDSCode",
        "PreschoolName"
    ])
)

# ---------------------------------------------------
# Enrollment supplemental fields
# ---------------------------------------------------

ce_fields = (
    ce
    .select(
        F.col("CSPPID").cast("string").alias("CSPPID"),
        F.col("ReportMonthNum").cast("int").alias("ReportMonthNum"),

        select_or_default(
            ce,
            "ProgramType",
            ["ProgramType"],
            "CSPP"
        ),

        select_or_default(
            ce,
            "ProgramDesignTypeCode",
            [
                "ProgramDesignTypeCode",
                "ProgramDesignType",
                "ProgramDuration",
                "ProgramDurationName",
                "ProgramDurationOffered1",
                "ProgramDurationOffered2"
            ],
            "UNKNOWN"
        ),

        select_or_default(
            ce,
            "FundingSourceCode",
            [
                "FundingSourceCode",
                "FundingSource",
                "FundingSourceName",
                "ContractType",
                "ContractTypeCode"
            ],
            "UNKNOWN"
        )
    )
    .dropDuplicates(["CSPPID", "ReportMonthNum"])
)

# ---------------------------------------------------
# LEFT JOIN preserves ChildFamily counts
# ---------------------------------------------------

base = (
    cf_base
    .join(
        ce_fields,
        on=["CSPPID", "ReportMonthNum"],
        how="left"
    )
    .withColumn(
        "ProgramType",
        F.coalesce(F.col("ProgramType"), F.lit("CSPP"))
    )
    .withColumn(
        "ProgramDesignTypeCode",
        F.coalesce(F.col("ProgramDesignTypeCode"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "FundingSourceCode",
        F.coalesce(F.col("FundingSourceCode"), F.lit("UNKNOWN"))
    )
)

# ---------------------------------------------------
# Monthly aggregation
# ---------------------------------------------------

monthly = (
    base
    .groupBy(
        "ReportMonthNum",
        "VendorNumber",
        "LEAName",
        "PreschoolCDSCode",
        "PreschoolName",
        "ProgramType",
        "ProgramDesignTypeCode",
        "FundingSourceCode"
    )
    .agg(
        F.countDistinct("CSPPID").alias("active_enrollment_count")
    )
)

# ---------------------------------------------------
# Lag features
# ---------------------------------------------------

w = Window.partitionBy(
    "VendorNumber",
    "LEAName",
    "PreschoolCDSCode",
    "PreschoolName",
    "ProgramType",
    "ProgramDesignTypeCode",
    "FundingSourceCode"
).orderBy("ReportMonthNum")

feat = (
    monthly
    .withColumn("lag_1", F.lag("active_enrollment_count", 1).over(w))
    .withColumn("lag_2", F.lag("active_enrollment_count", 2).over(w))
    .withColumn("lag_3", F.lag("active_enrollment_count", 3).over(w))
    .withColumn("target_next_month", F.lead("active_enrollment_count", 1).over(w))
)

for c in ["lag_1", "lag_2", "lag_3", "target_next_month"]:
    feat = feat.withColumn(c, F.col(c).cast("double"))

feat_clean = feat.dropna(
    subset=["lag_1", "lag_2", "lag_3", "target_next_month"]
)

assembler = VectorAssembler(
    inputCols=["lag_1", "lag_2", "lag_3"],
    outputCol="features",
    handleInvalid="skip"
)

data = assembler.transform(feat_clean)

train, test = data.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="target_next_month",
    predictionCol="prediction",
    numTrees=50,
    seed=42
)

model = rf.fit(train)

# ---------------------------------------------------
# Latest rows
# ---------------------------------------------------

latest_window = Window.partitionBy(
    "VendorNumber",
    "LEAName",
    "PreschoolCDSCode",
    "PreschoolName",
    "ProgramType",
    "ProgramDesignTypeCode",
    "FundingSourceCode"
).orderBy(F.col("ReportMonthNum").desc())

latest = (
    feat_clean
    .withColumn("rn", F.row_number().over(latest_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ---------------------------------------------------
# Recursive 5-month forecast
# ---------------------------------------------------

current = latest.select(
    "VendorNumber",
    "LEAName",
    "PreschoolCDSCode",
    "PreschoolName",
    "ProgramType",
    "ProgramDesignTypeCode",
    "FundingSourceCode",
    "ReportMonthNum",
    "active_enrollment_count",
    "lag_1",
    "lag_2",
    "lag_3"
)

all_forecasts = []

for step in range(1, 6):

    scored = (
        model.transform(assembler.transform(current))
        .withColumnRenamed("prediction", "predicted_enrollment")
        .withColumn(
            "ForecastReportMonth",
            add_months_yyyymm("ReportMonthNum", step)
        )
        .withColumn("forecast_horizon", F.lit(step))
        .withColumn(
            "trend_flag",
            F.when(
                F.col("predicted_enrollment") > F.col("active_enrollment_count"),
                F.lit("up")
            )
            .when(
                F.col("predicted_enrollment") < F.col("active_enrollment_count"),
                F.lit("down")
            )
            .otherwise(F.lit("flat"))
        )
        .select(
            "ForecastReportMonth",
            "forecast_horizon",
            "VendorNumber",
            "LEAName",
            "PreschoolCDSCode",
            "PreschoolName",
            "ProgramType",
            "ProgramDesignTypeCode",
            "FundingSourceCode",
            "active_enrollment_count",
            "lag_1",
            "lag_2",
            "lag_3",
            "predicted_enrollment",
            "trend_flag"
        )
    )

    all_forecasts.append(scored)

    current = (
        scored
        .withColumn(
            "ReportMonthNum",
            F.col("ForecastReportMonth")
        )
        .withColumn(
            "active_enrollment_count",
            F.col("predicted_enrollment")
        )
        .withColumn(
            "new_lag_1",
            F.col("predicted_enrollment")
        )
        .withColumn(
            "new_lag_2",
            F.col("lag_1")
        )
        .withColumn(
            "new_lag_3",
            F.col("lag_2")
        )
        .select(
            "VendorNumber",
            "LEAName",
            "PreschoolCDSCode",
            "PreschoolName",
            "ProgramType",
            "ProgramDesignTypeCode",
            "FundingSourceCode",
            "ReportMonthNum",
            "active_enrollment_count",
            F.col("new_lag_1").alias("lag_1"),
            F.col("new_lag_2").alias("lag_2"),
            F.col("new_lag_3").alias("lag_3")
        )
    )

forecast_5m = reduce(
    lambda a, b: a.unionByName(b),
    all_forecasts
)

# ---------------------------------------------------
# Summaries with names included
# ---------------------------------------------------

forecast_summary = (
    forecast_5m
    .groupBy("ForecastReportMonth")
    .agg(
        F.sum("predicted_enrollment")
        .alias("predicted_total_enrollment")
    )
)

forecast_vendor_summary = (
    forecast_5m
    .groupBy(
        "ForecastReportMonth",
        "VendorNumber",
        "LEAName"
    )
    .agg(
        F.sum("predicted_enrollment")
        .alias("predicted_vendor_enrollment")
    )
)

forecast_site_summary = (
    forecast_5m
    .groupBy(
        "ForecastReportMonth",
        "VendorNumber",
        "LEAName",
        "PreschoolCDSCode",
        "PreschoolName"
    )
    .agg(
        F.sum("predicted_enrollment")
        .alias("predicted_site_enrollment")
    )
)

# ---------------------------------------------------
# Write outputs
# ---------------------------------------------------

forecast_5m.write.mode("overwrite").parquet(
    FORECAST_DETAIL_OUT
)

forecast_summary.write.mode("overwrite").parquet(
    FORECAST_TOTAL_OUT
)

forecast_vendor_summary.write.mode("overwrite").parquet(
    FORECAST_VENDOR_OUT
)

forecast_site_summary.write.mode("overwrite").parquet(
    FORECAST_SITE_OUT
)

print("saved")
print(FORECAST_DETAIL_OUT)
print(FORECAST_TOTAL_OUT)
print(FORECAST_VENDOR_OUT)
print(FORECAST_SITE_OUT)


NameError: name 'spark' is not defined